# Steps 3-6 - Benchmark, Cross-Validation & Evaluation

Train and compare 15 classifiers across four families with an identical train/test strategy. Report stratified 5-fold cross-validation (mean / std / 95% CI) and a full held-out metric suite: Accuracy, Balanced Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, MCC, Cohen's kappa, Log-Loss, Brier, and train/predict times.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
from src import data, models, benchmark
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
zoo = models.get_model_zoo(x_train)
print("Models:", len(zoo))
for n, fam in models.MODEL_FAMILIES.items():
    print(f"  [{fam:9s}] {n}")

Models: 15
  [Linear   ] Logistic Regression
  [Linear   ] Ridge Classifier
  [Linear   ] SGD Classifier
  [Tree     ] Decision Tree
  [Tree     ] Random Forest
  [Tree     ] Extra Trees
  [Boosting ] AdaBoost
  [Boosting ] Gradient Boosting
  [Boosting ] XGBoost
  [Boosting ] LightGBM
  [Boosting ] CatBoost
  [Other    ] KNN
  [Other    ] GaussianNB
  [Other    ] SVM
  [Other    ] MLP


## 3.1 Stratified 5-fold cross-validation

In [3]:
cv = benchmark.cross_validate_models(x_train, y_train, zoo)
cv_view = cv[["Model", "F1_mean", "F1_std", "F1_ci_low", "F1_ci_high",
              "ROC AUC_mean", "Accuracy_mean", "MCC_mean"]].copy()
cv_view = cv_view.sort_values("F1_mean", ascending=False).round(4)
display(cv_view)
utils.save_table(cv.round(6), "cross_validation_full")
utils.save_table(cv_view, "cross_validation_summary",
                 caption="Stratified 5-fold CV: F1 mean, std and 95% CI (training set).",
                 label="tab:cv")

,Model,F1_mean,F1_std,F1_ci_low,F1_ci_high,ROC AUC_mean,Accuracy_mean,MCC_mean
4,Random Forest,0.9986,0.0010,0.9973,0.9998,1.0000,0.9988,0.9975
5,Extra Trees,0.9984,0.0010,0.9972,0.9996,1.0000,0.9986,0.9972
14,CatBoost,0.9982,0.0014,0.9965,1.0000,0.9998,0.9985,0.9969
12,XGBoost,0.9979,0.0013,0.9962,0.9995,0.9998,0.9982,0.9962
13,LightGBM,0.9972,0.0007,0.9963,0.9981,0.9999,0.9975,0.9950
7,Gradient Boosting,0.9968,0.0029,0.9933,1.0004,0.9993,0.9972,0.9944
3,Decision Tree,0.9965,0.0015,0.9946,0.9984,0.9968,0.9969,0.9937
11,MLP,0.9959,0.0010,0.9947,0.9972,0.9993,0.9965,0.9928
8,KNN,0.9952,0.0013,0.9936,0.9969,0.9986,0.9958,0.9916
10,SVM,0.9845,0.0019,0.9821,0.9870,0.9992,0.9863,0.9726


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/cross_validation_summary.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/cross_validation_summary.tex')}

**Observation.** CV F1 standard deviations are small for tree/boosting ensembles, indicating stable generalisation across folds. Linear models trail with wider spread, reflecting their inability to model the nonlinear class boundary.

## 3.2 Held-out test evaluation (full metric suite + timing)

In [4]:
results, fitted, predictions = benchmark.evaluate_on_test(x_train, y_train, x_test, y_test, zoo)
display(results.round(4))

# Persist the primary benchmark (Step 13).
utils.save_benchmark(results.round(6), "benchmark")
utils.save_table(results.round(4), "benchmark",
                 caption="Held-out test performance of all models.", label="tab:benchmark")
print("Saved benchmark.csv / benchmark.xlsx / benchmark.tex")

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC,PR AUC,MCC,Cohen Kappa,Log Loss,Brier Score,Train Time (s),Predict Time (s)
0,Extra Trees,0.9982,0.9980,0.9986,0.9972,0.9979,1.0000,1.0000,0.9962,0.9962,0.0043,0.0013,0.1552,0.0272
1,XGBoost,0.9982,0.9980,0.9986,0.9972,0.9979,0.9999,0.9999,0.9962,0.9962,0.0070,0.0016,0.4100,0.0041
2,CatBoost,0.9982,0.9980,0.9986,0.9972,0.9979,0.9999,0.9999,0.9962,0.9962,0.0066,0.0013,0.4248,0.0025
3,Decision Tree,0.9975,0.9975,0.9972,0.9972,0.9972,0.9975,0.9956,0.9950,0.9950,0.0851,0.0025,0.0103,0.0017
4,LightGBM,0.9969,0.9969,0.9958,0.9972,0.9965,0.9999,0.9999,0.9937,0.9937,0.0212,0.0030,0.6974,0.0062
5,Random Forest,0.9969,0.9966,0.9986,0.9944,0.9965,1.0000,1.0000,0.9937,0.9937,0.0060,0.0017,0.3415,0.0267
6,Gradient Boosting,0.9957,0.9955,0.9958,0.9944,0.9951,0.9995,0.9991,0.9912,0.9912,0.0200,0.0036,0.2615,0.0040
7,KNN,0.9957,0.9954,0.9972,0.9929,0.9951,0.9973,0.9962,0.9912,0.9912,0.0896,0.0040,0.0066,0.0258
8,MLP,0.9945,0.9940,0.9972,0.9901,0.9936,0.9992,0.9991,0.9887,0.9887,0.0265,0.0042,2.4499,0.0084
9,AdaBoost,0.9920,0.9919,0.9901,0.9915,0.9908,0.9983,0.9977,0.9837,0.9837,0.5529,0.1819,0.0895,0.0053


Saved benchmark.csv / benchmark.xlsx / benchmark.tex


## 3.3 Persist trained models and predictions

In [5]:
import joblib, numpy as np
for name, est in fitted.items():
    utils.save_model(est, name)
# Save test predictions for downstream notebooks (stats, error analysis).
pred_store = {name: {"y_pred": p["y_pred"],
                     "y_proba": p["y_proba"],
                     "y_score": p["y_score"]} for name, p in predictions.items()}
joblib.dump({"predictions": pred_store,
             "y_test": y_test.values,
             "x_test": x_test.reset_index(drop=True)},
            C.MODELS_DIR / "test_predictions.joblib")
print("Saved", len(fitted), "models to", C.rel(C.MODELS_DIR))

Saved 15 models to models


## 3.4 Visual comparison

In [6]:
for metric in ["F1", "ROC AUC", "MCC", "Balanced Accuracy"]:
    viz.plot_metric_bar(results, metric)
print("Saved comparison bar charts.")

Saved comparison bar charts.


## Summary\nTree ensembles and gradient boosters (Random Forest, Extra Trees, XGBoost, LightGBM, CatBoost) reach near-ceiling F1 and MCC, while linear models lag by 8-15 F1 points. The gap confirms a nonlinear decision boundary. Ceiling-level scores are examined critically in later notebooks (robustness, error, statistical significance).